In [1]:
# Install compatible version
!pip -q install transformers==4.52.4 sentencepiece

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# -----------------------------
# 1. Create a small dataset
# -----------------------------
texts = [
    "The transformer model achieved excellent accuracy.",
    "Large Language Models are revolutionizing AI.",
    "The football team won the championship.",
    "The cricket match was exciting.",
    "Neural networks are widely used in deep learning.",
    "The player scored a brilliant goal.",
    "Machine learning improves decision making.",
    "The tennis tournament starts tomorrow."
]

# 1 = Technology, 0 = Sports
labels = [1, 1, 0, 0, 1, 0, 1, 0]

# -----------------------------
# 2. Load tokenizer
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# -----------------------------
# 3. Create dataset class
# -----------------------------
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=64
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "labels": torch.tensor(self.labels[idx])
        }

dataset = TextDataset(texts, labels)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# -----------------------------
# 4. Load pre-trained BERT model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.to(device)

# -----------------------------
# 5. Optimizer
# -----------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# -----------------------------
# 6. Fine-tune the model
# -----------------------------
model.train()

epochs = 2

for epoch in range(epochs):
    total_loss = 0

    for batch in loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss = {total_loss:.4f}")

# -----------------------------
# 7. Save model
# -----------------------------
model.save_pretrained("fine_tuned_model")
tokenizer.save_pretrained("fine_tuned_model")

# -----------------------------
# 8. Load model for prediction
# -----------------------------
classifier = pipeline(
    "text-classification",
    model="fine_tuned_model",
    tokenizer="fine_tuned_model"
)

# -----------------------------
# 9. Test prediction
# -----------------------------
test_text = "Generative AI models improve intelligent automation."

result = classifier(test_text)

label_map = {
    "LABEL_0": "Sports",
    "LABEL_1": "Technology"
}

print("\nPrediction")
print("-" * 30)
print("Input :", test_text)
print("Predicted Class :", label_map[result[0]["label"]])
print("Confidence Score :", round(result[0]["score"], 3))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 100.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 Loss = 2.7943
Epoch 2 Loss = 2.0246


Device set to use cpu



Prediction
------------------------------
Input : Generative AI models improve intelligent automation.
Predicted Class : Technology
Confidence Score : 0.711
